# 学习rpy2

- 2605

# 什么是rpy2?

**rpy2** 是 Python 与 R 之间的**双向桥梁**，让你能在 Python 代码中直接调用 R 的功能。

---

## 核心定位

| 场景 | 解决方案 |
|------|---------|
| Python 做流程控制，但需要 DESeq2 做差异分析 | rpy2 调用 R 的 DESeq2 |
| R 画图好看，想在 Python 脚本里复用 ggplot2 | rpy2 运行 ggplot2 代码 |
| 已有大量 R 代码，想迁移到 Python 生态 | rpy2 逐步替换，无需重写 |

---

## 工作原理

```
Python 代码 → rpy2 接口 → R 解释器 → R 包/函数 → 结果返回 Python
     ↑_________________________________________________↓
```

rpy2 在 Python 进程中**嵌入了一个 R 引擎**，数据在两边实时转换。

---

## 基础用法示例

### 1. 安装

```bash
pip install rpy2
# 或
conda install -c conda-forge rpy2
```

### 2. 调用 R 函数

```python
import rpy2.robjects as ro

# 直接执行 R 代码
ro.r('''
    x <- c(1, 2, 3, 4, 5)
    mean(x)
''')

# 获取 R 变量到 Python
result = ro.r('mean(x)')
print(result[0])  # 3.0
```

### 3. 数据框互传（最常用）

```python
import pandas as pd
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

# 激活自动转换
pandas2ri.activate()

# Python DataFrame → R data.frame
df = pd.DataFrame({
    'gene': ['A', 'B', 'C'],
    'expr': [10, 20, 30]
})

# 调用 R 的 dplyr
dplyr = importr('dplyr')
r_df = pandas2ri.py2rpy(df)

# 在 R 中处理
ro.globalenv['df'] = r_df
result = ro.r('df %>% filter(expr > 15)')

# 转回 Python
python_df = pandas2ri.rpy2py(result)
```

### 4. 调用 Bioconductor 包（生信实战）

```python
from rpy2.robjects.packages import importr
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, Formula

pandas2ri.activate()

# 导入 R 包
deseq2 = importr('DESeq2')
base = importr('base')

# 准备数据（Python）
count_matrix = pd.DataFrame(...)  # 基因表达矩阵
metadata = pd.DataFrame(...)      # 样本信息

# 转为 R 对象
r_counts = pandas2ri.py2rpy(count_matrix)
r_meta = pandas2ri.py2rpy(metadata)

# 构建 DESeqDataSet
dds = deseq2.DESeqDataSetFromMatrix(
    countData=r_counts,
    colData=r_meta,
    design=Formula('~ condition')  # 实验设计
)

# 运行 DESeq2
dds = deseq2.DESeq(dds)
res = deseq2.results(dds, contrast=ro.StrVector(['condition', 'treat', 'ctrl']))

# 结果转回 Python DataFrame
result_df = pandas2ri.rpy2py(base.as_data_frame(res))
```

---

## 在 `omics4plant` 中的角色

```
┌─────────────────────────────────────┐
│         Python 主流程控制            │
│  (omics4plant/core/pipeline.py)     │
└─────────────┬───────────────────────┘
              │ 调用
┌─────────────▼───────────────────────┐
│      rpy2 桥接层                    │
│  (omics4plant/analysis/differential.py) │
│  - 封装 DESeq2 / edgeR / limma      │
│  - 封装 clusterProfiler             │
│  - 封装 ggplot2 绘图                │
└─────────────┬───────────────────────┘
              │ 调用
┌─────────────▼───────────────────────┐
│         R 生物信息包生态             │
│  DESeq2, edgeR, limma,              │
│  clusterProfiler, enrichplot, etc.  │
└─────────────────────────────────────┘
```

---

## 优缺点速览

| ✅ 优势 | ❌ 局限 |
|--------|---------|
| 复用 R 庞大的生信包生态 | 需要系统安装 R，环境配置复杂 |
| 数据自动转换（pandas ↔ data.frame） | 调试困难，R 报错信息难定位 |
| 无需重写成熟的统计方法 | 性能 overhead，大数据量时较慢 |
| 可以逐步迁移，平滑过渡 | R 包版本兼容性需管理 |

---

## 替代方案对比

| 方案 | 原理 | 适用场景 |
|------|------|---------|
| **rpy2** | 进程内嵌入 R | 频繁交互、需实时数据传递 |
| **subprocess 调用 Rscript** | 命令行执行 R 脚本 | 简单任务、一次调用 |
| **纯 Python 重写** | 用 scanpy/pydeseq2 等 | 长期维护、追求性能 |

---

## 实际建议

对于你的 `omics4plant` 生态：

1. **核心统计方法**（DESeq2、edgeR、WGCNA）→ **rpy2 桥接**，不要重复造轮子
2. **流程控制和数据管理** → **纯 Python**，发挥 Python 的工程优势
3. **新出现的 Python 原生包**（如 `pydeseq2`、`scanpy`）→ **逐步替换**，降低 rpy2 依赖

rpy2 的本质是**让你站在 R 生态的肩膀上，用 Python 的方式飞行**。

# rpy2如何链接R环境

rpy2 默认会自动寻找系统 R，但指定特定 Conda 环境中的 R 需要手动配置。以下是针对你的场景（`/opt/conda/env/seurat/bin/R` 中的 SoupX）的完整方案：

---

## 核心原理

rpy2 通过环境变量找到 R 解释器，关键变量是：

| 环境变量 | 作用 |
|---------|------|
| `R_HOME` | R 的安装根目录 |
| `LD_LIBRARY_PATH` | R 共享库路径（Linux） |

---

## 方案一：启动 Python 前指定环境（推荐）

```bash
# 1. 激活目标 Conda 环境（获取正确路径）
conda activate seurat
which R  # 确认路径：/opt/conda/env/seurat/bin/R

# 2. 获取 R_HOME
R RHOME  # 输出如：/opt/conda/env/seurat/lib/R

# 3. 启动 Python 时注入环境变量
export R_HOME=/opt/conda/env/seurat/lib/R
export LD_LIBRARY_PATH=/opt/conda/env/seurat/lib/R/lib:$LD_LIBRARY_PATH
export PATH=/opt/conda/env/seurat/bin:$PATH

# 4. 启动你的 Python
python your_script.py
```

---

## 方案二：Python 代码内动态切换（灵活）

```python
import os
import sys

# ========== 在导入 rpy2 之前配置 ==========

# 指定目标 R 环境
R_HOME = "/opt/conda/env/seurat/lib/R"
R_BIN = "/opt/conda/env/seurat/bin/R"

# 设置环境变量（必须在 import rpy2 之前）
os.environ["R_HOME"] = R_HOME
os.environ["PATH"] = f"/opt/conda/env/seurat/bin:{os.environ.get('PATH', '')}"

# Linux 需要 R 共享库
if sys.platform == "linux":
    r_lib_path = f"{R_HOME}/lib"
    current_ld = os.environ.get("LD_LIBRARY_PATH", "")
    if r_lib_path not in current_ld:
        os.environ["LD_LIBRARY_PATH"] = f"{r_lib_path}:{current_ld}"

# ========== 现在导入 rpy2 ==========
import rpy2.robjects as ro
from rpy2.robjects.packages import importr

# 验证链接的 R 版本
print(ro.r("R.version.string")[0])
# 应输出类似：[1] "R version 4.3.1 (2023-06-16)"

# ========== 调用 SoupX ==========
# SoupX 是 R 包，通过 rpy2 导入
soupx = importr("SoupX")

# 示例：运行 SoupX 基本流程
ro.r("""
    library(SoupX)
    
    # 你的 SoupX 分析代码
    # sc = load10X(dataDir)
    # sc = setContaminationFraction(sc, 0.2)
    # out = adjustCounts(sc)
""")

# 或者逐行调用
sc = ro.r('load10X("/path/to/10x_data")')
sc = ro.r('setContaminationFraction')(sc, 0.2)
out = ro.r('adjustCounts')(sc)
```

---

## 方案三：封装为可复用模块（推荐用于 omics4plant）

```python
# omics4plant/core/r_env_manager.py
import os
import sys
from pathlib import Path
from typing import Optional


class REnvManager:
    """
    管理多个 R 环境的切换
    """
    
    def __init__(self, r_home: Optional[str] = None):
        self.r_home = r_home or self._auto_detect()
        self._original_env = {}
        self._save_original()
    
    def _auto_detect(self) -> str:
        """尝试自动发现 R"""
        # 优先级：环境变量 > which R > 常见路径
        if "R_HOME" in os.environ:
            return os.environ["R_HOME"]
        
        # 尝试找到 R 可执行文件
        import shutil
        r_bin = shutil.which("R")
        if r_bin:
            # 从 bin/R 推导 lib/R
            return str(Path(r_bin).parent.parent / "lib" / "R")
        
        raise RuntimeError("无法自动检测 R 环境，请手动指定 R_HOME")
    
    def _save_original(self):
        """保存原始环境变量"""
        for key in ["R_HOME", "LD_LIBRARY_PATH", "PATH"]:
            self._original_env[key] = os.environ.get(key)
    
    def activate(self):
        """激活指定 R 环境"""
        # R 可执行文件路径
        r_bin_dir = str(Path(self.r_home).parent.parent / "bin")
        
        os.environ["R_HOME"] = self.r_home
        
        # 更新 PATH
        current_path = os.environ.get("PATH", "")
        if r_bin_dir not in current_path:
            os.environ["PATH"] = f"{r_bin_dir}:{current_path}"
        
        # Linux 共享库
        if sys.platform == "linux":
            r_lib = f"{self.r_home}/lib"
            current_ld = os.environ.get("LD_LIBRARY_PATH", "")
            if r_lib not in current_ld:
                os.environ["LD_LIBRARY_PATH"] = f"{r_lib}:{current_ld}"
        
        # 关键：清除 rpy2 的缓存，强制重新初始化
        self._clear_rpy2_cache()
        
        print(f"✓ R 环境已激活: {self.r_home}")
        return self
    
    def _clear_rpy2_cache(self):
        """清除 rpy2 模块缓存，允许重新加载"""
        modules_to_remove = [
            k for k in sys.modules.keys() 
            if k.startswith("rpy2")
        ]
        for mod in modules_to_remove:
            del sys.modules[mod]
    
    def verify(self) -> str:
        """验证 R 版本"""
        import rpy2.robjects as ro
        version = ro.r("R.version.string")[0]
        return version
    
    def restore(self):
        """恢复原始环境"""
        for key, value in self._original_env.items():
            if value is not None:
                os.environ[key] = value
            elif key in os.environ:
                del os.environ[key]


# ========== 使用示例 ==========

# 场景1：使用 seurat 环境的 SoupX
r_env = REnvManager(r_home="/opt/conda/env/seurat/lib/R")
r_env.activate()
print(r_env.verify())  # R version 4.x.x

from rpy2.robjects.packages import importr
soupx = importr("SoupX")

# 场景2：切换到另一个环境的 Monocle3
r_env2 = REnvManager(r_home="/opt/conda/env/monocle/lib/R")
r_env2.activate()
monocle = importr("monocle3")
```

---

## 方案四：直接在目标环境中运行（最简单）

如果你不需要频繁切换 R 环境，最直接的方式：

```bash
# 激活 seurat 环境（里面有 R + SoupX）
conda activate seurat

# 在该环境中安装 Python 和 rpy2
conda install python rpy2

# 直接运行，无需任何额外配置
python -c "
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
soupx = importr('SoupX')
print('SoupX 版本:', ro.r('packageVersion(\"SoupX\")')[0])
"
```

---

## 常见问题排查

| 问题 | 原因 | 解决 |
|------|------|------|
| `R_HOME not set` | rpy2 找不到 R | 先设置 `R_HOME` 再 `import rpy2` |
| `cannot find -lR` | 缺少 R 共享库 | 设置 `LD_LIBRARY_PATH` |
| `SoupX not found` | R 包未安装在目标环境 | 在 seurat 环境中 `R -e "install.packages('SoupX')"` |
| `version mismatch` | rpy2 与 R 版本不兼容 | 确保 rpy2 是在目标 R 环境中安装的 |
| `libreadline.so.6: cannot open` | 缺少系统依赖 | `conda install -c conda-forge readline` |

## 如何用rpy2跑soupx

```python
# 在分析模块中这样封装 SoupX
# omics4plant/analysis/singlecell.py

from omics4plant.core.r_env_manager import REnvManager


class SoupXProcessor:
    """
    基于 rpy2 的 SoupX 去污染封装
    """
    
    def __init__(self, r_home: str = "/opt/conda/env/seurat/lib/R"):
        self.r_env = REnvManager(r_home=r_home)
        self.r_env.activate()
        
        # 延迟导入，确保环境已激活
        import rpy2.robjects as ro
        from rpy2.robjects.packages import importr
        
        self.ro = ro
        self.soupx = importr("SoupX")
    
    def run(self, data_dir: str, contamination: float = 0.2):
        """
        运行 SoupX 去污染
        
        Args:
            data_dir: 10x 数据目录
            contamination: 预估污染比例
        """
        sc = self.ro.r('load10X')(data_dir)
        sc = self.ro.r('setContaminationFraction')(sc, contamination)
        out = self.ro.r('adjustCounts')(sc)
        return out
    
    def to_anndata(self, soupx_result):
        """将 SoupX 结果转为 Python anndata"""
        import anndata as ad
        import numpy as np
        
        # 从 R 提取矩阵
        counts = self.ro.r('as.matrix')(soupx_result)
        # 转为 numpy → anndata
        ...
        return adata


# 使用
processor = SoupXProcessor()
adata = processor.run("/path/to/10x_data", contamination=0.2)
```